In [ ]:
import os
import math
import csv
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Any, Dict, Counter, List
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv, CacheUnitMapper
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

import sys
# sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
# from Common.Utils import save_training_results


In [ ]:
# --- 1. CONFIGURATION & HYPERPARAMETERS (Section VII-B) ---
class Config:
    n_episodes: int = 1000
    n_nodes: int = 3
    n_users: int = 200
    step_size: float = 10.0
    arrival_rate: float = 10.0  # users per second
    zipf_alpha: float = 0.5
    n_videos: int = 500
    n_gops: int = 30
    n_layers: int = 2
    n: int = 4
    m: int = 3
    n_tiles: int = n * m
    tiles_per_viewport: int = 4

    base_tile_b = 2e6 / n_tiles
    enh_tile_b = 15e6 / n_tiles

    max_capacity: float = 500e6  # 500 MB
    cache_capacity_percent: float = 0.15  # 15% of the total video size
    cache_size: int = int(n_videos * cache_capacity_percent)
    cache_capacity_b: float = (
        n_gops * n_tiles * base_tile_b +
        n_gops * tiles_per_viewport * enh_tile_b
    ) * cache_size

    # Hyperparameters for RL
    epsilon_start: float = 1.0
    epsilon_min: float = 0.05
    epsilon_decay: float = (epsilon_min / epsilon_start) ** (1.0 / n_episodes) # 0.987

    gamma: float = 0.6 
    learning_rate: float = 1e-3
    batch_size: int = 32
    buffer_capacity: int = 2000
    nb_interval: int = 200  # train every 5 requests

    h_short: int = 300   # sliding windows for popularity (Section VI-A)
    h_long: int = 1000

    r_base: float = 30.0 # PSNR reward for base layer (Section VI-C)
    r_enh: float = 10.0  # PSNR reward for enhancement layer
    penalty: float = 0.0 # fetch penalty (implicit in paper)

    # CPT parameters
    theta: float = 0.5
    lam: float = 3.7183

    @property
    def state_dim(self) -> int: # 10*C + 2 = (2C + 2Ck) * 2 + 2 (Section VI-A)
        return 10 * self.cache_size + 2

    @property
    def action_dim(self) -> int: # |A| = 5C + 1 (Section VI-B)
        return (self.cache_size + self.cache_size * self.tiles_per_viewport + 1)

    @property
    def action_dim_2(self) -> int: # |A| = 5C + 1 (Section VI-B)
        return (self.cache_size + 1) * (self.cache_size * self.tiles_per_viewport + 1)

In [ ]:
cfg = Config()
action_dim = cfg.action_dim_2

C = cfg.cache_size
k = cfg.tiles_per_viewport
a2_size = C * k + 1

# action_idx in [0, (C+1)*(C*k+1) - 1]
action_idx = random.randint(0, action_dim - 1)
a1, a2 = divmod(action_idx, a2_size)

# decode a2 into (video_slot, tile_slot) if a2 > 0
if a2 == 0:
    video_slot = None
    tile_slot = None  # no-op
else:
    video_slot, tile_slot = divmod(a2 - 1, k)  # 0..C-1, 0..k-1
print(video_slot, tile_slot)

In [ ]:
# --- DataClass for User Transition ---
@dataclass
class UserTransition:
    state: Any
    action: int
    reward: float

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DQN(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        hidden = action_dim  # = 5C + 1
        self.fc1 = nn.Linear(state_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)  # linear

class ReplayBuffer:
    def __init__(self, capacity: int = 2000):
        self.memory = deque(maxlen=capacity)
    def push(self, s, a, r, ns, d):
        self.memory.append((s, a, r, ns, d))
    def sample(self, batch_size: int):
        return random.sample(self.memory, batch_size)
    def __len__(self):
        return len(self.memory)

class DQNAgent:
    def __init__(self, cfg: Config):
        self.state_dim = cfg.state_dim
        self.action_dim = cfg.action_dim_2
        self.epsilon = cfg.epsilon_start
        self.epsilon_min = cfg.epsilon_min
        self.epsilon_decay = cfg.epsilon_decay
        self.gamma = cfg.gamma
        self.batch_size = cfg.batch_size
        self.buffer = ReplayBuffer(cfg.buffer_capacity)
        self.policy_net = DQN(self.state_dim, self.action_dim).to(device)
        self.target_net = DQN(self.state_dim, self.action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=1e-3)
        self.loss_fn = nn.MSELoss()
        self.nb_interval = cfg.nb_interval  # train every 5 requests

        self.capacity = cfg.cache_size

    def _decode_action(self, action_idx: int) -> tuple[int, int]:
        a2_size = self.C * self.k + 1
        return action_idx // a2_size, action_idx % a2_size


    def select_action(self, state, idx: int = 0, g: int = 0):        
        if random.random() < self.epsilon:
            if g == 0:
                return random.randint(0, self.capacity), None
            else:
                offset = self.capacity + idx * 4 + 1
                action = random.randint(offset, offset + 3)

                return 0 if action == offset + 4 else action, None

        state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = self.policy_net(state)

        if g == 0:
            return q_values[0, 0:self.capacity + 1].argmax().item(), q_values
        else:
            offset = self.capacity + idx * 4 + 1
            slice_vals = q_values[0, offset : offset + 4]
            max_slice, max_idx = slice_vals.max(0)
            action = 0 if q_values[0, 0] >= max_slice else (offset + max_idx.item())

            return action, q_values

    def select_action_2(self, state, g: int, idx: int, cfg: Config):
        C = cfg.cache_size
        k = cfg.tiles_per_viewport
        a2_size = C * k + 1

        if random.random() < self.epsilon:
            action_idx = random.randint(0, self.action_dim - 1)
            a1, a2 = divmod(action_idx, a2_size)
            if a2 == 0:
                a1, a2 = 0, 0  # no-op
            else:
                a1, a2 = divmod(a2 - 1, k)  # 0..C-1, 0..k-1

            if g == 0:
                return a1, None
            else:
                return a2, None
        state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = self.policy_net(state)
        
        action_idx = q_values.argmax().item()
        a1, a2 = divmod(action_idx, a2_size)
        if a2 == 0:
            a1, a2 = 0, 0  # no-op
        else:
            a1, a2 = divmod(a2 - 1, k)  # 0..C-1, 0..k-1

        if g == 0:
            return a1, q_values
        else:
            return a2, q_values
    
    def remember(self, s, a, r, ns, done):
        self.buffer.push(s, a, r, ns, done)

    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return
        batch = self.buffer.sample(self.batch_size)
        s, a, r, ns, d = zip(*batch)
        s = torch.tensor(np.stack(s), dtype=torch.float32).to(device)
        ns = torch.tensor(np.stack(ns), dtype=torch.float32).to(device)
        a = torch.tensor(a, dtype=torch.int64).to(device)
        r = torch.tensor(r, dtype=torch.float32).to(device)
        d = torch.tensor(d, dtype=torch.float32).to(device)

        q = self.policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            q_next = self.target_net(ns).max(1)[0]
            target = r + self.gamma * q_next * (1.0 - d)

        loss = self.loss_fn(q, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def update_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

In [ ]:
def save_training_results(
    path_,
    filename,
    ep, 
    total_reward, 
    cache_hits, 
    cache_misses, 
    soft_hits,
    agent
):
    with open(os.path.join(path_, filename), 'a', newline='') as f:
        fieldnames = [
            'episode', 
            'total_reward', 
            'cache_hits', 
            'cache_misses', 
            'soft_hits',
            'epsilon'
        ]
        writer_results = csv.DictWriter(f, fieldnames=fieldnames)

        if ep == 0:
            writer_results.writeheader()
        
        writer_results.writerow({
            'episode': ep,
            'total_reward': total_reward,
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'soft_hits': soft_hits,
            'epsilon': agent.epsilon if agent else None
        })


In [ ]:
class FeatureAdapter:
    def __init__(self, env: CacheEngineEnv, cfg: Config):
        self.env = env
        self.cfg = cfg

        self.video_hist_short = deque(maxlen=cfg.h_short)
        self.video_hist_long = deque(maxlen=cfg.h_long)
        self.tile_hist_short = deque(maxlen=cfg.h_short)
        self.tile_hist_long = deque(maxlen=cfg.h_long)
        self.tiles_hist_short = deque(maxlen=cfg.h_short)
        self.tiles_hist_long = deque(maxlen=cfg.h_long)

        self.video_freq_short = defaultdict(int)
        self.video_freq_long = defaultdict(int)
        self.tile_freq_short = defaultdict(int)
        self.tile_freq_long = defaultdict(int)
        self.tiles_freq_short = defaultdict(int)
        self.tiles_freq_long = defaultdict(int)

    def reset_history(self):
        queues = (
            self.video_hist_short,
            self.video_hist_long,
            self.tiles_hist_short,
            self.tiles_hist_long,
            self.tile_hist_short,
            self.tile_hist_long,
        )
        freqs = (
            self.video_freq_short,
            self.video_freq_long,
            self.tiles_freq_short,
            self.tiles_freq_long,
            self.tile_freq_short,
            self.tile_freq_long,
        )
        
        for q in queues:
            q.clear()
        for f in freqs:
            f.clear()

    def update_history(self, vid: int, tiles: list[int]):       
        self._update_window(self.video_hist_short, self.video_freq_short, vid)
        self._update_window(self.video_hist_long, self.video_freq_long, vid)

        tiles = tuple(tiles) if tiles is not None else None

        if tiles is None:
            return

        self._update_window(self.tiles_hist_short, self.tiles_freq_short, tiles)
        self._update_window(self.tiles_hist_long, self.tiles_freq_long, tiles)

        for tile in tiles:
            self._update_window(self.tile_hist_short, self.tile_freq_short, (vid, tile))
            self._update_window(self.tile_hist_long, self.tile_freq_long, (vid, tile))

    def _update_window(self, hist_queue: deque, freq_dict: Dict, item):
        if len(hist_queue) == hist_queue.maxlen:
            old_item = hist_queue.popleft()
            freq_dict[old_item] -= 1
            if freq_dict[old_item] == 0:
                del freq_dict[old_item]
        hist_queue.append(item)
        freq_dict[item] += 1

In [ ]:
class NetworkAdapter:
    def __init__(self, env: EnvWrapper, feature_adapter: FeatureAdapter, cfg: Config):
        self.env = env
        self.cfg = cfg
        self.features = feature_adapter

        self.capacity = int(
            self.cfg.n_videos * self.cfg.cache_capacity_percent * self.cfg.n_gops * self.cfg.n_tiles +
            self.cfg.n_videos * self.cfg.cache_capacity_percent * self.cfg.n_gops * self.cfg.tiles_per_viewport            
        )
        self.n_features = self.capacity + 1

        self.C = self.cfg.cache_size               # paper’s cache capacity (videos)
        self.k = self.env.mec_cache.get_viewport_tile_budget()

        self.video_cache_index = [-1] * self.C
        self.tile_cache_index = [[-1] * self.k for _ in range(self.C)]

        print(f"NetworkAdapter initialized with capacity: {self.C} videos, {self.k} tiles per video")

    def _cache_video(self, vid, bitmap: np.ndarray) -> None:
        bitmap[vid, 0, :, :] = 1

    def _evict_video(self, vid: int, bitmap: np.ndarray) -> None:
        bitmap[vid, :, :, :] = 0
    
    def _evict_tile(self, vid: int, tile: int, bitmap: np.ndarray) -> None:
        bitmap[vid, 1, tile, :] = 0

    def has_video_base_layer(self, vid: int) -> bool:
        return vid in self.video_cache_index

    def get_video_cache_idx(self, vid: int) -> int:
        for idx, v in enumerate(self.video_cache_index):
            if v == vid:
                return idx
        return -1

    def build_observation(self, vid: int, gop: int, viewport: List[int]) -> np.ndarray:
        """
        Builds the state vector as in the paper:
          [ x_s (C), y_s (C*k), z_s (1), x_l (C), y_l (C*k), z_l (1) ]
        where:
          - x_s/x_l: counts of requests for cached base videos (short/long windows)
          - y_s/y_l: counts of requests for cached enh tiles per video (short/long)
          - z_s/z_l: counts for the currently examined item (video or tile)
        Total dim = 10*C + 2 when k=4.
        """
        x_s = np.zeros(self.C, dtype=np.float32)
        x_l = np.zeros(self.C, dtype=np.float32)

        y_s = np.zeros(self.C * self.k, dtype=np.float32)
        y_l = np.zeros(self.C * self.k, dtype=np.float32)

        for vid_i, v in enumerate(self.video_cache_index):
            if v == -1:
                continue
            x_s[vid_i] = self.features.video_freq_short.get(v, 0)
            x_l[vid_i] = self.features.video_freq_long.get(v, 0)

            tiles = self.tile_cache_index[vid_i]

            for til_i, t in enumerate(tiles):
                if t == -1:
                    continue
                y_s[vid_i * self.k + til_i] = self.features.tile_freq_short.get((v, t), 0)
                y_l[vid_i * self.k + til_i] = self.features.tile_freq_long.get((v, t), 0)

        z_s = np.array([
            self.features.video_freq_short.get(vid, 0) if gop == 0 else 
            sum(self.features.tile_freq_short.get((vid, t), 0) for t in viewport)
        ], dtype=np.float32)
        z_l = np.array([
            self.features.video_freq_long.get(vid, 0) if gop == 0
            else sum(self.features.tile_freq_long.get((vid, t), 0) for t in viewport)
        ], dtype=np.float32)

        return np.concatenate([x_s, x_l, y_s, y_l, z_s, z_l], axis=0)

    def apply_video_action(self, action_idx: int, vid: int) -> Dict:
        """
        Implements the paper's action space:
          - A1 (size C+1): when base is not cached. a0 = no-op; a_i evicts the i-th cached video and caches the requested one.
          - A2 (size k+1): when base is cached but viewport differs. a0 = no-op; a_j replaces the j-th cached enh tile with the j-th requested tile.
        """
        if action_idx == 0:
            return  # No-op

        if action_idx <= self.C:
            self.video_cache_index[action_idx - 1] = vid
            self.tile_cache_index[action_idx - 1] = [-1] * self.k
            
        # print(f"Video cache index after action: {self.video_cache_index}")

    def apply_tile_action(self, action_idx: int, vid: int, tile: int, gop: int):
        # if action_idx == 0:
        #     return  # No-op

        vid_idx = self.get_video_cache_idx(vid)
        tile_idx = action_idx

        if tile not in self.tile_cache_index[vid_idx]:
            self.tile_cache_index[vid_idx][tile_idx] = tile

    def apply_tile_action_2(self, action_idx: int, vid: int, tile: int, gop: int):
        if action_idx == 0:
            return  # No-op

        vid_idx = self.get_video_cache_idx(vid)
        
        tile_block_start = self.C + vid_idx * self.k
        tile_idx = action_idx - (tile_block_start + 1) # 0..3

        if tile not in self.tile_cache_index[vid_idx]:
            self.tile_cache_index[vid_idx][tile_idx] = tile

    def last_sample_replication(
        self, 
        vid: int, 
        gop: int, 
        viewport: list[int],
    ):
        vid_idx = self.get_video_cache_idx(vid)

        for i, tile in enumerate(viewport):
            self.tile_cache_index[vid_idx][i] = int(tile)

    def calc_cache_hits(
        self, 
        vid: int, 
        gop: int,
        viewport: list[int], 
    ) -> tuple[int, int, float]:

        vid_idx = self.get_video_cache_idx(vid)

        if gop == 0 and vid_idx == -1:
            return 0, 12, 0.0
        elif gop == 0 and vid_idx != -1:
            return 12, 0, 30.0

        if vid_idx == -1:
            return 0, 12, 0.0

        hits = 12  # base layer hit
        misses = 0
        distortion = 30.0

        cached_tiles = self.tile_cache_index[vid_idx]
        
        for t_idx in viewport:
            if t_idx in cached_tiles:
                hits += 1
                distortion += 2.5
            else:
                misses += 1

        return hits, misses, distortion

    def get_next_user_request(self, u: int, gop: int, cfg: Config) -> list[int]:
        if gop + 1 >= self.env.users_env.n_gops:
            return np.array([], dtype=int)
        
        current_viewport = self.env.users_env.users_viewport_tiles[u][gop+1]
        return np.array(
            [y * cfg.n + x for x, y in current_viewport], dtype=int
        )
    
    def reset(self):
        self.video_cache_index = [-1] * self.C
        self.tile_cache_index = [[-1] * self.k for _ in range(self.C)]

        return self.env.reset()
    
    def env_is_done(self) -> bool:
        return self.env.users_env.users_done >= self.env.users_env.n_users

In [ ]:
if __name__ == "__main__":
    print("--- Starting DRL Caching System ---")

    # 1. Load Configuration
    cfg = Config()

    # 2. Initialize Environment
    du_caches = []

    unit_mapper = CacheUnitMapper(
        cache_capacity_mb=cfg.cache_capacity_b / 1e6,
        num_gops=cfg.n_gops,
        num_tiles=cfg.n_tiles,
        viewport_tiles=4,  # assuming viewport with 4 tiles
        base_tile_mb=2e6 / 1e6 / cfg.n_tiles,
        enh_tile_mb=15e6 / 1e6 / cfg.n_tiles
    )

    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.cache_capacity_b,
        unit_mapper=unit_mapper
    )

    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=cfg.arrival_rate,
        alpha=cfg.zipf_alpha
    )

    P = cfg.n_nodes; max_U = cfg.n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,  # 640 Mbps -> 80e6 B/s
        R_C_M=125e6, # 1 Gbps -> 125e6 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),    # 160 Mbps -> 20e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,  # 1 ms
        mec_fixed_delay=0.005, # 5 ms
        cloud_fixed_delay=0.1  # 100 ms
    )

    env = EnvWrapper(
        n=cfg.n,
        n_layers=cfg.n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        theta=cfg.theta,
        lam=cfg.lam
    )

    # 3. Initialize Agent
    agent = DQNAgent(cfg)

    obs, info = env.reset()
    feature_adapter = FeatureAdapter(env, cfg)
    net_adapter = NetworkAdapter(env, feature_adapter, cfg)

    for episode in range(cfg.n_episodes):

        obs, info = net_adapter.reset()
        feature_adapter.reset_history()
        
        cache_hits = 0
        cache_misses = 0
        soft_hits = 0.0
        total_reward = 0.0
        avg_psnr = []

        user_history = {u: 0.0 for u in range(cfg.n_users)}

        for step in count():

            user_transition: Dict[int, UserTransition] = {}

            # Get active users (not finished all GOPs)
            reqs_state = info['users_requests']

            # Main Loop. Process each active user request
            for req in reqs_state:
                u, p, v, g, tiles = req['u'], req['p'], req['video'], req['gop'], req["tiles"]
                viewport = req['viewport'] if req['viewport'] is not None else []

                has_base_layer = net_adapter.has_video_base_layer(v)
                action_taken = False

                # print(f"Processing User {u}, Video {v}, Gop {g}, Viewport {viewport} - Has Base Layer: {has_base_layer}")
                if g == 0 and not has_base_layer:
                    ch, cm, ds = net_adapter.calc_cache_hits(v, g, viewport)

                    # print(f"User {u}, Video {v}, Gop {g}, Viewport {viewport} - Caching base layer")
                    state = net_adapter.build_observation(v, g, viewport)
                    a_idx, _ = agent.select_action_2(state, g, 0, cfg)

                    net_adapter.apply_video_action(a_idx, v)
                    action_taken = True

                elif g > 0 and has_base_layer:
                    ch, cm, ds = net_adapter.calc_cache_hits(v, g, viewport)

                    idx = net_adapter.get_video_cache_idx(v)
                    cached_tiles = set(net_adapter.tile_cache_index[idx])

                    missing_tiles = [t for t in viewport if t not in cached_tiles]

                    # print(f"User {u}, Video {v}, Gop {g}, Viewport {viewport}, Missing Tiles: {missing_tiles}")

                    if len(missing_tiles) > 0:
                        state = net_adapter.build_observation(v, g, viewport)
                        for t in missing_tiles:
                            a_idx, _ = agent.select_action_2(state, g, idx, cfg)
                            net_adapter.apply_tile_action(a_idx, v, t, g)

                        action_taken = True
                else:
                    ch, cm, ds = net_adapter.calc_cache_hits(v, g, viewport)

                # Update feature history for the requested video and viewport
                feature_adapter.update_history(v, viewport)

                user_history[u] += ds
                cache_hits += ch
                cache_misses += cm

                if action_taken:
                    user_transition[u] = UserTransition(
                        state=state, 
                        action=a_idx, 
                        reward=user_history[u]
                    )
                    total_reward += user_history[u]

            # Updates in the state after processing all active users
            reqs_next_state = net_adapter.env.users_env.step(
                None, None
            )
            reqs_next_by_user = {r["u"]: r for r in reqs_next_state}

            for req in reqs_state:
                u, v, g = req['u'], req['video'], req['gop']
                viewport = req['viewport'] if req['viewport'] is not None else []

                if u not in reqs_next_by_user:
                    continue

                if u not in user_transition:
                    continue

                req_next = reqs_next_by_user[u]
                next_state = net_adapter.build_observation(v, g, viewport)

                # --- TRAINING / HISTORY UPDATE ---
                if u in user_transition:
                    agent.remember(
                        user_transition[u].state, 
                        user_transition[u].action, 
                        user_transition[u].reward, 
                        next_state, 
                        done=net_adapter.env.users_env.user_is_done(u)
                    )

            if step % agent.nb_interval == 0:
                agent.train_step()
                agent.update_target()

            if net_adapter.env_is_done():
                break

            info = {
                'users_requests': reqs_next_state
            }

            print(f"Step {step}, Active Users: {len(reqs_state)}")
            # print(f"Request State: {reqs_state}")
            # print(f"Next Request State: {reqs_next_state}")
            print("----------------------------------------------------------------")

        agent.update_epsilon()

        filename = (
            f"drl_pan_E{cfg.n_episodes}_U{cfg.n_users}_"
            f"V{cfg.n_videos}_G{cfg.n_gops}_L{cfg.n_layers}_"
            f"cap{cfg.cache_size}_AR{cfg.arrival_rate}_Z{cfg.zipf_alpha}.csv"
        )

        save_training_results(
            # path_='/home/eduardo/Workspace/CacheVideoPredict360/Results',
            path_=r'c:\Users\es25591\Workspace\CacheVideoPredict360\Results',
            filename=filename,
            ep=episode,
            total_reward=total_reward,
            cache_hits=cache_hits,
            cache_misses=cache_misses,
            soft_hits=soft_hits,
            agent=agent
        )

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------------------------------------------      
# 1. Publication Style Configuration
# ----------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True
})

# Professional color for line plots
PRIMARY_COLOR = "#2b7bba"

# ----------------------------------------------------------------
# 2. Data Loading & Smoothing
# ----------------------------------------------------------------
path = r"c:\Users\es25591\Workspace\CacheVideoPredict360\Results\drl_pan_c50_AR10.0_Z0.5.csv"
df = pd.read_csv(path)

# Metrics to plot
metrics = ["total_reward", "cache_hits", "cache_misses", "epsilon"]
window_size = 10  # Adjust smoothing window as needed

# ----------------------------------------------------------------
# 3. Plotting logic
# ----------------------------------------------------------------
# Adjusted figsize for a 4-column row (standard for full-width paper figures)
fig, axes = plt.subplots(1, len(metrics), figsize=(12, 3), sharex=True)

for ax, col in zip(axes, metrics):
    # Plot raw data with transparency (alpha)
    ax.plot(df["episode"], df[col], color=PRIMARY_COLOR, alpha=0.3, linewidth=0.8, label='Raw')
    
    # Plot moving average for clearer trend (except for epsilon which is usually linear)
    if col != "epsilon":
        smoothed = df[col].rolling(window=window_size).mean()
        ax.plot(df["episode"], smoothed, color=PRIMARY_COLOR, linewidth=1.5, label='Trend')
    else:
        # Just a solid line for Epsilon
        ax.plot(df["episode"], df[col], color=PRIMARY_COLOR, linewidth=1.5)

    # Stylistic cleanup
    ax.set_title(col.replace("_", " ").title(), fontweight="bold")
    ax.set_xlabel("Episode")
    
    # Remove redundant Y-labels to save space, or keep for clarity
    ax.set_ylabel("Value") 
    
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    
    # Tufte-style: remove top/right spines
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

# Optional: Add a single legend to the first plot if needed
# axes[0].legend(frameon=False)

plt.show()
fig.savefig("drl_caching_metrics.png", dpi=300)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------------------------------------------      
# 1. Publication Style Configuration
# ----------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True
})

COLORS = ["#2b7bba", "#d62728"]

# ----------------------------------------------------------------
# 2. Data Preparation with Variance
# ----------------------------------------------------------------
def get_metrics_with_error(path):
    """Returns mean and standard deviation for hit and miss rates."""
    try:
        df = pd.read_csv(path)
        total = (df["cache_hits"] + df["cache_misses"]).replace(0, np.nan)
        
        hit_series = (df["cache_hits"] / total * 100)
        miss_series = (df["cache_misses"] / total * 100)
        
        # Calculate Mean and Standard Deviation
        return (hit_series.mean(), hit_series.std()), (miss_series.mean(), miss_series.std())
    except Exception as e:
        print(f"Error processing {path}: {e}")
        return (0, 0), (0, 0)

drl_files = {
    "5": r"c:\Users\es25591\Workspace\CacheVideoPredict360\Results\drl_pan_opt_c25_ar10.0_z0.5.csv",
    "10": r"c:\Users\es25591\Workspace\CacheVideoPredict360\Results\drl_pan_opt_c50_ar10.0_z0.5.csv",
}

lsr_files = {
    "5": r"c:\Users\es25591\Workspace\CacheVideoPredict360\Results\lru_c25_ar10.0_z0.5.csv",
    "10": r"c:\Users\es25591\Workspace\CacheVideoPredict360\Results\lru_c50_ar10.0_z0.5.csv",
}

caps = list(drl_files.keys())
# Structure: { Metric: { Algo: [means], Algo_err: [stds] } }
stats = {
    "Hit Rate": {"DRL-LSR": [], "DRL-LSR_err": [], "LRU-LSR": [], "LRU-LSR_err": []},
    "Miss Rate": {"DRL-LSR": [], "DRL-LSR_err": [], "LRU-LSR": [], "LRU-LSR_err": []}
}

for c in caps:
    (d_h, d_h_err), (d_m, d_m_err) = get_metrics_with_error(drl_files[c])
    (l_h, l_h_err), (l_m, l_m_err) = get_metrics_with_error(lsr_files[c])
    
    stats["Hit Rate"]["DRL-LSR"].append(d_h)
    stats["Hit Rate"]["DRL-LSR_err"].append(d_h_err)
    stats["Hit Rate"]["LRU-LSR"].append(l_h)
    stats["Hit Rate"]["LRU-LSR_err"].append(l_h_err)
    
    stats["Miss Rate"]["DRL-LSR"].append(d_m)
    stats["Miss Rate"]["DRL-LSR_err"].append(d_m_err)
    stats["Miss Rate"]["LRU-LSR"].append(l_m)
    stats["Miss Rate"]["LRU-LSR_err"].append(l_m_err)

# ----------------------------------------------------------------
# 3. Plotting logic
# ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(7, 3.2), sharey=True)
x = np.arange(len(caps))
width = 0.35 

# Error bar styling
error_kw = dict(lw=1, capsize=3, capthick=1, ecolor='#333333')

metrics = ["Hit Rate", "Miss Rate"]
for i, metric in enumerate(metrics):
    ax = axes[i]
    
    ax.bar(x - width/2, stats[metric]["DRL-LSR"], width, 
           yerr=stats[metric]["DRL-LSR_err"], error_kw=error_kw,
           label="DRL-LSR", color=COLORS[0], edgecolor='black', linewidth=0.6, zorder=3)
    
    ax.bar(x + width/2, stats[metric]["LRU-LSR"], width, 
           yerr=stats[metric]["LRU-LSR_err"], error_kw=error_kw,
           label="LRU-LSR", color=COLORS[1], edgecolor='black', linewidth=0.6, zorder=3)

    ax.set_title(f"Average {metric}", fontweight="bold")
    ax.set_xlabel("Cache Size (%)")
    ax.set_xticks(x)
    ax.set_xticklabels(caps)
    ax.set_ylim(0, 110) # Increased to accommodate error bars
    
    ax.grid(axis='y', linestyle='--', alpha=0.3, zorder=0)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

axes[0].set_ylabel("Percentage (%)")
axes[1].legend(frameon=False, loc="upper right")

plt.tight_layout()
plt.show()

fig.savefig("drl_vs_lru_cache_performance.pdf")